<div style="background: linear-gradient(135deg, #1f1c2c 0%, #928DAB 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 2</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Kafka CLI Fundamentals</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Master the Kafka Command Line Interface: Topics, Partitions, Offsets, Producers, and Consumers.</p>
</div>

---

## 🎯 Overview

In this lab, you will learn how to interact with a Kafka cluster using its native Command Line Interface (CLI) tools. These tools are essential for administration, debugging, and day-to-day operations.

By the end of this lab, you will understand how to:
- 📋 **Create, List, and Describe** Kafka topics
- 📤 **Produce** messages to a topic (with and without keys)
- 📥 **Consume** messages and understand message formatting
- 🧩 Understand **Partitions** and how messages are routed
- 📍 Understand **Offsets** and Consumer Groups

---

## ⚙️ Prerequisites

We assume your Kafka cluster from **Lab 1** is already running. We will use the `kafka_training-kafka-1` container to execute CLI commands.

---

## <span style="color: #928DAB;">Step 1:</span> Verify Cluster Status

Let's ensure your Kafka cluster containers are up and running before we begin executing CLI commands.

In [1]:
!docker ps --filter "name=kafka" 

CONTAINER ID   IMAGE                                 COMMAND                  CREATED         STATUS                   PORTS                                         NAMES
037d48744d0a   confluentinc/cp-kafka-connect:7.5.0   "/etc/confluent/dock…"   4 minutes ago   Up 4 minutes (healthy)   0.0.0.0:8083->8083/tcp, [::]:8083->8083/tcp   kafka-connect
29e97a755af3   confluentinc/cp-kafka:7.5.0           "/etc/confluent/dock…"   4 minutes ago   Up 4 minutes             0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp   kafka


---

## <span style="color: #928DAB;">Step 2:</span> Create a Topic

The `kafka-topics` script is used to manage topics. Let's create a new topic named `orders`.

**Key arguments:**
- `--bootstrap-server`: The address of the Kafka broker to connect to.
- `--create`: The action we want to perform.
- `--topic`: The name of the topic.
- `--partitions`: The number of partitions (determines scalability and parallel processing).
- `--replication-factor`: The number of copies of the data (fault tolerance). Since we have 1 broker, this must be 1.

In [2]:
!docker exec kafka kafka-topics \
  --bootstrap-server localhost:9092 \
  --create \
  --topic orders \
  --partitions 3 \
  --replication-factor 1

Created topic orders.


---

## <span style="color: #928DAB;">Step 3:</span> List Topics

Let's verify that the `orders` topic was successfully created by listing all available topics in the cluster.

In [3]:
!docker exec kafka kafka-topics \
  --bootstrap-server localhost:9092 \
  --list

__consumer_offsets
_confluent-command
_confluent-controlcenter-7-5-0-1-AlertHistoryStore-changelog
_confluent-controlcenter-7-5-0-1-AlertHistoryStore-repartition
_confluent-controlcenter-7-5-0-1-Group-ONE_MINUTE-changelog
_confluent-controlcenter-7-5-0-1-Group-ONE_MINUTE-repartition
_confluent-controlcenter-7-5-0-1-Group-THREE_HOURS-changelog
_confluent-controlcenter-7-5-0-1-Group-THREE_HOURS-repartition
_confluent-controlcenter-7-5-0-1-KSTREAM-OUTEROTHER-0000000106-store-changelog
_confluent-controlcenter-7-5-0-1-KSTREAM-OUTEROTHER-0000000106-store-repartition
_confluent-controlcenter-7-5-0-1-KSTREAM-OUTERTHIS-0000000105-store-changelog
_confluent-controlcenter-7-5-0-1-KSTREAM-OUTERTHIS-0000000105-store-repartition
_confluent-controlcenter-7-5-0-1-MetricsAggregateStore-changelog
_confluent-controlcenter-7-5-0-1-MetricsAggregateStore-repartition
_confluent-controlcenter-7-5-0-1-MonitoringMessageAggregatorWindows-ONE_MINUTE-changelog
_confluent-controlcenter-7-5-0-1-MonitoringMessageAgg

---

## <span style="color: #928DAB;">Step 4:</span> Describe a Topic

Describing a topic shows its underlying configuration, particularly its partitions.

**Understanding the Output:**
- **Partition**: The ID of the partition (e.g., 0, 1, 2).
- **Leader**: The broker node currently acting as the leader for this partition.
- **Replicas**: The brokers holding a copy of this partition.
- **Isr**: The "In-Sync Replicas" - brokers that are fully caught up with the leader.

In [2]:
!docker exec kafka kafka-topics \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic orders

Error response from daemon: No such container: kafka


---

## <span style="color: #928DAB;">Step 5:</span> Produce Messages (Without Keys)

The `kafka-console-producer` tool reads from standard input and sends messages to a topic. 
Here, we're piping three order messages directly into the producer.

Because we do **not** specify a message key, Kafka will distribute these messages across the 3 partitions using a **Round-Robin** or sticky partitioning strategy.

In [3]:
!docker exec kafka bash -c "echo -e 'Order 1\nOrder 2\nOrder 3' | kafka-console-producer --bootstrap-server localhost:9092 --topic orders"

Error response from daemon: No such container: kafka


---

## <span style="color: #928DAB;">Step 6:</span> Consume Messages

The `kafka-console-consumer` tool reads data from a topic and outputs it to standard out.

We use the `--from-beginning` flag to tell Kafka to read all messages from offset 0 of every partition. *(Note: We use `--timeout-ms` here so the notebook cell doesn't hang indefinitely)*.

In [9]:
!docker exec kafka kafka-console-consumer \
  --bootstrap-server localhost:9092 \
  --topic orders \
  --from-beginning \
  --timeout-ms 1000

Order 1
Order 2
Order 3


[2026-05-17 14:13:01,955] ERROR Error processing message, terminating consumer process:  (kafka.tools.ConsoleConsumer$)
org.apache.kafka.common.errors.TimeoutException
Processed a total of 3 messages


---

## <span style="color: #928DAB;">Step 7:</span> Produce Messages (With Keys)

In a real-world scenario, messages usually have a **Key** (e.g., Customer ID, Order ID). 
Kafka guarantees that **messages with the same key always go to the same partition**. This guarantees order per key.

Let's produce messages formatted as `key:value`.

In [10]:
!docker exec kafka bash -c "echo -e 'userA:Order 4\nuserB:Order 5\nuserA:Order 6' | kafka-console-producer \
  --bootstrap-server localhost:9092 \
  --topic orders \
  --property 'parse.key=true' \
  --property 'key.separator=:'"

In [1]:
!docker exec kafka kafka-console-consumer \
  --bootstrap-server localhost:9092 \
  --topic <your-topic-name> \
  --from-beginning \
  --property print.key=true \
  --property print.partition=true \
  --property key.separator=" | "


The system cannot find the file specified.


---

## 🏆 Student Challenge

Time to test your new skills! Try to complete the following tasks without looking back at the previous steps.

**Your Mission:**
1. Create a new topic named `logs` with **2 partitions**.
2. Produce **4 messages** into the `logs` topic: 
   - Two messages with the key `error`
   - Two messages with the key `info`
3. Consume all messages from the `logs` topic and verify that all `error` messages went to the exact same partition.

<div style="background-color: rgba(231, 76, 60, 0.1); border-left: 4px solid #e74c3c; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>💡 Hint:</strong> Remember to use <code>--property parse.key=true</code> and <code>--property key.separator=:</code> when producing messages with keys!
</div>

In [ ]:
# 👩‍💻 Write your CLI commands here! 
# (Feel free to create more code cells if you want to run them one by one)

!docker exec kafka_training-kafka-1 echo "Ready for the challenge!"

---

## <span style="color: #928DAB;">Step 11:</span> Clean Up (Deleting Topics)

Finally, let's clean up our workspace by deleting the `orders` topic.

In [ ]:
!docker exec kafka_training-kafka-1 kafka-topics \
  --bootstrap-server localhost:9092 \
  --delete \
  --topic orders

<div style="background-color: rgba(146, 141, 171, 0.1); border: 1px solid rgba(146, 141, 171, 0.3); padding: 20px; text-align: center; border-radius: 8px; margin-top: 40px;">
  <h3 style="color: #928DAB; margin-bottom: 10px;">🎉 Lab 2 Complete!</h3>
  <p style="color: #8b949e; margin: 0;">You are now proficient with the Kafka Command Line Interface and understand the mechanics of topics, partitions, and offsets.</p>
</div>